# NYC Weather Analysis for Small Business Optimization - Phase 2

**Main Question:**
- How can small businesses in NYC optimize operations by analyzing weather patterns to predict short-term disruptions?
## Phase 1
###  Sub-Questions:
        1. What are optimal delivery/operation hours based on weather conditions?
        2. What weather conditions constitute a 'disruption' for small business operations?
        3. Which days pose the highest operational risk due to weather?
### Result:
    Deterministic model that allow to identify if there was a dispruption or not based on three main variables.
---

    **Data Source:** Open-Meteo Historical Weather API  
    **Location:** New York City (40.738°N, 74.043°W)  
    **Period:** June 1 - March 15, 2026
    **Granularity:** 15 minutely measurements

    **Weather Variables:**
        - Temperature (°F)
        - Relative Humidity (%)
        - Precipitation (inches)
        - Rain (inches)
        - Wind Speed (mph)
        - Wind Gusts (mph)
---
## Phase 2:
**Objective**:
- Improve the deterministic model by including other weather variables into the analysis and train a predictive model to allow small businesses to react accordingly.
---

    **Data Source:** Open-Meteo Historical Weather API  
    **Location:** New York City (40.738°N, 74.043°W)  
    **Period:** Most recent data
    **Granularity:** 15 minutely measurements
    **Weather Variables:**
        - Temperature (°F)
        - Relative Humidity (%)
        - Showers
        - Snofall
        - Surface pressure
        - Cloude Cover
        - Precipitation (inches)
        - Rain (inches)
        - Wind Speed (mph)
        - Wind Gusts (mph)
---


---
## Part 1: Setup and Data Loading

This section initializes Spark with MinIO S3 configuration and loads the raw weather data.

In [ ]:
# import required libraries
from pyspark.sql import SparkSession

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd

In [ ]:
# Initialize Spark Session with MinIO S3A Configuration

spark = SparkSession.builder \
    .appName("NYC Weather Analysis") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "osbdet") \
    .config("spark.hadoop.fs.s3a.secret.key", "osbdet123$") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk-bundle:1.11.1026") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print(f"Spark Session Created {spark.version}")

26/03/18 11:31:05 WARN Utils: Your hostname, osbdet resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/18 11:31:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/osbdet/.jupyter_venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/osbdet/.ivy2/cache
The jars for the packages stored in: /home/osbdet/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9c954af3-5d47-4ec6-a7c9-7e866b630a14;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
:: resolution report :: resolve 1839ms :: artifacts dl 53ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.11.1026] in [default]
	-----------------------------------------------------------

Spark Session Created 3.5.4


In [ ]:
# Load raw JSON data from MinIO
# The data is stored as a single JSON file with parallel arrays structure

df_raw = spark.read.json("s3a://raw-weather/new_york_weather_historical_data.json")

print("Raw data loaded from MinIO")
print(f"Columns: {df_raw.columns}")
df_raw.printSchema()

26/03/18 11:31:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/18 11:31:40 WARN VersionInfoUtils: The AWS SDK for Java 1.x entered maintenance mode starting July 31, 2024 and will reach end of support on December 31, 2025. For more information, see https://aws.amazon.com/blogs/developer/the-aws-sdk-for-java-1-x-is-in-maintenance-mode-effective-july-31-2024/
You can print where on the file system the AWS SDK for Java 1.x core runtime is located by setting the AWS_JAVA_V1_PRINT_LOCATION environment variable or aws.java.v1.printLocation system property to 'true'.
This message can be disabled by setting the AWS_JAVA_V1_DISABLE_DEPRECATION_ANNOUNCEMENT environment variable or aws.java.v1.disableDeprecationAnnouncement system property to 'true'.
The AWS SDK for Java 1.x is being used here:
at java.base/java.lang.Thread.getStackTrace(Thread.java:2451)
at com.amazonaws.util.VersionInfoUtils.printDeprecationAn

Raw data loaded from MinIO
Columns: ['elevation', 'generationtime_ms', 'latitude', 'longitude', 'minutely_15', 'minutely_15_units', 'timezone', 'timezone_abbreviation', 'utc_offset_seconds']
root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- minutely_15: struct (nullable = true)
 |    |-- apparent_temperature: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- cloud_cover: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- precipitation: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- rain: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- showers: array (nullable = true)
 |    |    |-- element: double (contai

---
## Part 2: Data Transformation

The raw JSON has weather variables stored as parallel arrays (e.g., all temperatures in one array, all timestamps in another).  
We need to transform this into a tabular format where each row represents one hour of data.

In [ ]:
# Parse parallel arrays into tabular format using arrays_zip and explode

# Why arrays_zip? It combines multiple arrays element-by-element into an array of structs
# Why explode? It converts the array of structs into individual rows

# Example: ["a","b"], [1,2], [10,20] = {time:"a", temp:1, wind:10}, {time:"b", temp:2, wind:20}
# Its indexing the first positon and assigning it the correct value inside of the dictionary

df_weather = df_raw.select(
    explode(
        arrays_zip(
            col("minutely_15.time"),
            col("minutely_15.temperature_2m"),
            col("minutely_15.apparent_temperature"),
            col("minutely_15.cloud_cover"),
            col("minutely_15.showers"),
            col("minutely_15.snowfall"),
            col("minutely_15.surface_pressure"),
            col("minutely_15.relative_humidity_2m"),
            col("minutely_15.precipitation"),
            col("minutely_15.rain"),
            col("minutely_15.wind_speed_10m"),
            col("minutely_15.wind_gusts_10m")
        )
    ).alias("hourly_data")
).select(
    # Access struct fields by their column names (not indices!)
    col("hourly_data.time").alias("timestamp"),
    col("hourly_data.temperature_2m").alias("temperature"),
    col("hourly_data.apparent_temperature").alias("apparent_temperature"),
    col("hourly_data.cloud_cover").alias("cloud_cover"),
    col("hourly_data.snowfall").alias("snowfall"),
    col("hourly_data.showers").alias("showers"),
    col("hourly_data.surface_pressure").alias("surface_pressure"),
    col("hourly_data.relative_humidity_2m").alias("humidity"),
    col("hourly_data.precipitation").alias("precipitation"),
    col("hourly_data.rain").alias("rain"),
    col("hourly_data.wind_speed_10m").alias("windspeed"),
    col("hourly_data.wind_gusts_10m").alias("windgusts")
)


# df_weather = df_weather.filter(df_weather.timestamp > "2026-01-7:00")

print("Data transformed into tabular format")
print(f"Total records: {df_weather.count()}")

# checking a sample of 5
print("\nSample data:")
df_weather.show(5, truncate=False)


Data transformed into tabular format


Total records: 27648

Sample data:


+----------------+-----------+--------------------+-----------+--------+-------+----------------+--------+-------------+----+---------+---------+
|timestamp       |temperature|apparent_temperature|cloud_cover|snowfall|showers|surface_pressure|humidity|precipitation|rain|windspeed|windgusts|
+----------------+-----------+--------------------+-----------+--------+-------+----------------+--------+-------------+----+---------+---------+
|2025-06-01T00:00|53.8       |45.4                |9          |0.0     |0.0    |996.0           |57      |0.0          |0.0 |22.2     |38.5     |
|2025-06-01T00:15|53.5       |46.1                |26         |0.0     |0.0    |996.1           |57      |0.0          |0.0 |18.1     |40.3     |
|2025-06-01T00:30|53.1       |45.6                |49         |0.0     |0.0    |996.1           |58      |0.0          |0.0 |18.4     |42.5     |
|2025-06-01T00:45|52.8       |45.2                |70         |0.0     |0.0    |996.2           |58      |0.0          |0.0 

In [ ]:
from pyspark.sql import functions as F

df_weather.select( [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_weather.columns] ).toPandas()

,timestamp,temperature,apparent_temperature,cloud_cover,snowfall,showers,surface_pressure,humidity,precipitation,rain,windspeed,windgusts
0,0,0,0,0,0,0,0,0,0,0,0,0


---
## Part 3: Feature Engineering - Defining Disruption Criteria

### What weather conditions constitute a 'disruption'?

Using the deterministic model we define disruptions based on industry standards and NYC operational experience for the historical data, this gives us the input that is going to be used to train a model that should find out the importance of other variables when predicting the disruptions:

**Disruption Thresholds:**
1. **Heavy Rain:** Precipitation > 0.15 inches/hour
   - Reasoning: Reduces visibility, causes traffic delays, unsafe for outdoor work
2. **High Wind:** Wind speed > 18 mph
   - Reasoning: Dangerous for deliveries, equipment damage risk, pedestrian safety
3. **Extreme Temperature:** < 40°F (freezing) or > 95°F (extreme heat)
   - Reasoning: Equipment failure, worker safety, customer behavior changes

A disruption is flagged if **ANY** of these conditions occur.

In [ ]:
# Create derived columns for time analysis and disruption detection
# Changing everything to datetime, timestamp, for easier analysis with pandas
df_analyzed = df_weather \
    .withColumn("datetime", to_timestamp("timestamp")) \
    .withColumn("date", to_date("datetime")) \
    .withColumn("hour", hour("datetime")) \
    .withColumn("day_of_week", dayofweek("datetime")) \
    .withColumn("month", month("datetime")) \
    .withColumn("month_name", date_format("datetime", "MMMM")) \
    .withColumn("week_of_year", weekofyear("datetime")) \
    .withColumn("heavy_rain", when(col("precipitation") > 0.15, 1).otherwise(0)) \
    .withColumn("high_wind", when(col("windspeed") > 18, 1).otherwise(0)) \
    .withColumn("extreme_temp", when((col("temperature") < 40) | (col("temperature") > 95), 1).otherwise(0)) \
    .withColumn("disruption",
        when((col("heavy_rain") == 1) | (col("high_wind") == 1) | (col("extreme_temp") == 1), 1)
        .otherwise(0)
    ) \
    .withColumn("temp_category",
        when(col("temperature") < 40, "Freezing")
        .when(col("temperature") < 50, "Cold")
        .when(col("temperature") < 70, "Moderate")
        .when(col("temperature") < 85, "Warm")
        .otherwise("Hot")
    ) \
    .withColumn("day_name",
        when(col("day_of_week") == 1, "Sunday")
        .when(col("day_of_week") == 2, "Monday")
        .when(col("day_of_week") == 3, "Tuesday")
        .when(col("day_of_week") == 4, "Wednesday")
        .when(col("day_of_week") == 5, "Thursday")
        .when(col("day_of_week") == 6, "Friday")
        .when(col("day_of_week") == 7, "Saturday")
    )

# Creating the criteria for the different columns (criteria = disruptive or not) using CASE-style logic
# Column labeled as disruption if any of these conditions are met (| == or)
# Creating a categorical column temp_category for the different temp types (Fahrenheit btw because 'merica)
# Creating new column with the day of the week

print("\nDisruption flags created:")
df_analyzed = df_analyzed.select('hour','day_of_week','month','temperature','apparent_temperature','cloud_cover','snowfall','showers','surface_pressure','humidity','precipitation','rain','windspeed','windgusts','disruption')
df_analyzed.show(10)


Disruption flags created:


+----+-----------+-----+-----------+--------------------+-----------+--------+-------+----------------+--------+-------------+----+---------+---------+----------+
|hour|day_of_week|month|temperature|apparent_temperature|cloud_cover|snowfall|showers|surface_pressure|humidity|precipitation|rain|windspeed|windgusts|disruption|
+----+-----------+-----+-----------+--------------------+-----------+--------+-------+----------------+--------+-------------+----+---------+---------+----------+
|   0|          1|    6|       53.8|                45.4|          9|     0.0|    0.0|           996.0|      57|          0.0| 0.0|     22.2|     38.5|         1|
|   0|          1|    6|       53.5|                46.1|         26|     0.0|    0.0|           996.1|      57|          0.0| 0.0|     18.1|     40.3|         1|
|   0|          1|    6|       53.1|                45.6|         49|     0.0|    0.0|           996.1|      58|          0.0| 0.0|     18.4|     42.5|         1|
|   0|          1|    

In [ ]:
df_analyzed.groupby('disruption').count().show()

[Stage 9:>                                                          (0 + 1) / 1]

+----------+-----+
|disruption|count|
+----------+-----+
|         1|11836|
|         0|15812|
+----------+-----+



# Part 4: Model Design and Training
With the historical data we trained two models to predict disruptions: a Logistic Regression model and a Random Forest model.

In [ ]:
seed=123

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import RandomForestClassifier


lr = LogisticRegression(labelCol="disruption", featuresCol="features")
rf = RandomForestClassifier(labelCol="disruption", featuresCol="features",maxDepth=10,seed=seed)


classifiers = [lr, rf]
classifiers

[LogisticRegression_3bfbba46b6ed, RandomForestClassifier_7c85a38119df]

In [ ]:
# Create pipelines to define the steps for the Model

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler

label_col = 'disruption'

# stringIndexer = StringIndexer(inputCol='disruption', outputCol="label_idx", handleInvalid="keep")

# Preparation of the Feature Transformation PySpark object to consolidate everything in one single vector
assemblerInputs = [c for c in df_analyzed.columns if c not in [label_col, "label_idx"]]

vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

def create_pipeline(classifier):
    return Pipeline(stages = [ vecAssembler, classifier])

pipelines = [create_pipeline(classifier) for classifier in classifiers]

pipelines

[Pipeline_18060ca78626, Pipeline_14e17e6075e9]

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="disruption",  metricName="weightedPrecision", metricLabel= 1.0)

In [ ]:
# In order to have two sets, we're going to split the dataset like this

(trainingData1_df, testData1_df) = df_analyzed.where("disruption=0").randomSplit([0.8,0.2],seed=seed)
(trainingData2_df, testData2_df) = df_analyzed.where("disruption=1").randomSplit([0.8,0.2],seed=seed)

trainingData_df = trainingData1_df.unionByName(trainingData2_df)
testData_df = testData1_df.unionByName(testData2_df)
trainingData_df.count()
testData_df.count()

22027

5621

In [ ]:
# Let's train all the models at once

models = [pipeline.fit(trainingData_df) for pipeline in pipelines]
models

26/03/18 11:35:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/18 11:35:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
                                                                                

[PipelineModel_956c2669754a, PipelineModel_dfd6be62efef]

# Part 5: Model Selection

In [ ]:

names = []
values_training = []
values_test = []

for model in models:
    prediction_training = model.transform(trainingData_df)
    prediction_test = model.transform(testData_df)
    precision_training = evaluator.evaluate(prediction_training)
    precision_test = evaluator.evaluate(prediction_test)
    names.append(type(model.stages[-1]).__name__) # the algorithm is the last stage in the pipeline
    values_training.append(precision_training)
    values_test.append(precision_test)


data = {'name': names, 'Precision_training': values_training,"Precision_test":values_test,'model': models}
df = pd.DataFrame(data)
df.sort_values(by=['Precision_test'], inplace=True, ascending=False)
df

,name,Precision_training,Precision_test,model
1,RandomForestClassificationModel,0.988841,0.985264,PipelineModel_dfd6be62efef
0,LogisticRegressionModel,0.902285,0.902076,PipelineModel_956c2669754a


In [ ]:
best_model=df.iloc[0]['model']

# Part 6: Disruption Prediction

Having the model trained, the small businesses in the city can now predict if there will be a disruption in their service during the next hour due to weather conditions. The API used provides the weather conditions every 15 minutes, so the businesses can predict if there will be a disruption within the next hour of service.

## Data Loading of the most recent weather information

In [ ]:
df_current_raw = spark.read.json("s3a://raw-weather/new_york_weather_current.json")

print("Raw data loaded from MinIO")
print(f"Columns: {df_raw.columns}")
df_current_raw.printSchema()

Raw data loaded from MinIO
Columns: ['elevation', 'generationtime_ms', 'latitude', 'longitude', 'minutely_15', 'minutely_15_units', 'timezone', 'timezone_abbreviation', 'utc_offset_seconds']
root
 |-- current: struct (nullable = true)
 |    |-- apparent_temperature: double (nullable = true)
 |    |-- cloud_cover: long (nullable = true)
 |    |-- interval: long (nullable = true)
 |    |-- precipitation: double (nullable = true)
 |    |-- rain: double (nullable = true)
 |    |-- relative_humidity_2m: long (nullable = true)
 |    |-- showers: double (nullable = true)
 |    |-- snowfall: double (nullable = true)
 |    |-- surface_pressure: double (nullable = true)
 |    |-- temperature_2m: double (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- wind_gusts_10m: double (nullable = true)
 |    |-- wind_speed_10m: double (nullable = true)
 |-- current_units: struct (nullable = true)
 |    |-- apparent_temperature: string (nullable = true)
 |    |-- cloud_cover: string (null

## Data Transformation and Feature Engineering

In [ ]:
from pyspark.sql.functions import col, to_timestamp, to_date, dayofweek, month, hour, when

df_current_weather = df_current_raw.select(
    col("current.time").alias("timestamp"),
    col("current.temperature_2m").alias("temperature"),
    col("current.apparent_temperature").alias("apparent_temperature"),
    col("current.cloud_cover").alias("cloud_cover"),
    col("current.snowfall").alias("snowfall"),
    col("current.showers").alias("showers"),
    col("current.surface_pressure").alias("surface_pressure"),
    col("current.relative_humidity_2m").alias("humidity"),
    col("current.precipitation").alias("precipitation"),
    col("current.rain").alias("rain"),
    col("current.wind_speed_10m").alias("windspeed"),
    col("current.wind_gusts_10m").alias("windgusts")
)

df_analyzed = df_current_weather \
    .withColumn("datetime", to_timestamp("timestamp")) \
    .withColumn("date", to_date("datetime")) \
    .withColumn("hour", hour("datetime")) \
    .withColumn("day_of_week", dayofweek("datetime")) \
    .withColumn("month", month("datetime")) \
    .withColumn(
        "day_of_week_name",
        when(col("day_of_week") == 1, "Sunday")
        .when(col("day_of_week") == 2, "Monday")
        .when(col("day_of_week") == 3, "Tuesday")
        .when(col("day_of_week") == 4, "Wednesday")
        .when(col("day_of_week") == 5, "Thursday")
        .when(col("day_of_week") == 6, "Friday")
        .when(col("day_of_week") == 7, "Saturday")
    )

df_analyzed = df_analyzed.select(
    "timestamp",
    "hour",
    "day_of_week",
    "month",
    "temperature",
    "apparent_temperature",
    "cloud_cover",
    "snowfall",
    "showers",
    "surface_pressure",
    "humidity",
    "precipitation",
    "rain",
    "windspeed",
    "windgusts"
)

## Prediction

In [ ]:
predictions = best_model.transform(df_analyzed)

In [ ]:
predictions.select("timestamp", "prediction").toPandas()

26/03/18 11:38:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

,timestamp,prediction
0,2026-03-18T06:30,1.0


In [ ]:
predictions.toPandas()

,timestamp,hour,day_of_week,month,temperature,apparent_temperature,cloud_cover,snowfall,showers,surface_pressure,humidity,precipitation,rain,windspeed,windgusts,features,rawPrediction,probability,prediction
0,2026-03-18T06:30,6,4,3,27.1,17.6,0,0.0,0.0,1024.1,48,0.0,0.0,12.7,19.8,"[6.0, 4.0, 3.0, 27.1, 17.6, 0.0, 0.0, 0.0, 102...","[0.0, 20.0]","[0.0, 1.0]",1.0


### ===============================================END OF NOTEBOOK===============================================